# This file contains functions to train the machine learning models on the archive.org cookbooks.

# 0. Imports

In [2]:
import os
import re
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import joblib
from sklearn.svm import SVC
from xgboost import XGBClassifier
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score



/Users/yavuzlule/Desktop/bsc-thesis-copy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Auxiliary functions

In [1]:

def construct_dataset_from_directory(dir_name):
    """
    Assumes directory has two subfolders: 'cooking' and 'other', each with text files.
    Returns a pandas DataFrame with columns ['text', 'label'].
    """
    data = []
    for label_name, label in [('cooking', 1), ('other', 0)]:
        folder_path = os.path.join(dir_name, label_name)
        if not os.path.exists(folder_path):
            continue
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if os.path.isfile(file_path) and filename.endswith('.txt'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    text = f.read()
                    data.append({'text': text, 'label': label})
    df = pd.DataFrame(data)
    return df

def save_dataset_to_directory(df, dir_name):
    os.makedirs(dir_name, exist_ok=True)
    file_path = os.path.join(dir_name, 'dataset.csv')
    df.to_csv(file_path, index=False)
    print(f"Dataset saved to {file_path}")

def get_embeddings(df, sentence_embedding_model):
    """
    df: pandas DataFrame with 'text' column
    sentence_embedding_model: instance of SentenceTransformer
    Returns: numpy array of embeddings
    """
    texts = df['text'].tolist()
    embeddings = sentence_embedding_model.encode(texts, show_progress_bar=True)
    return embeddings

def train_model(model, dataset):
    """
    model: sklearn model (LogisticRegression, SVM with probability, XGBoost, etc.)
    dataset: pandas DataFrame with 'text' and 'label'
    Returns trained model
    """
    X = dataset['mini_lm_embeddings'].tolist()  # embeddings must be precomputed and added to df
    y = dataset['is_cooking'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model.fit(X_train, y_train)
    print(f"Model training completed. Test accuracy: {model.score(X_test, y_test):.4f}")
    return model

def save_model(model, dir_name, model_name):
    """
    Saves a trained model to disk with a custom name.
    """
    os.makedirs(dir_name, exist_ok=True)
    model_path = os.path.join(dir_name, f"{model_name}.joblib")
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path}")
    

def preprocess_text(text):
    if text is None:
        return ""

    # lowercase
    text = text.lower()

    # normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # remove multiple empty lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    # collapse repeated spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # remove strange OCR artifacts (common in scanned books)
    text = re.sub(r"[^\w\s\n.,:;!?()\-/]", "", text)

    return text.strip()

def divide_books_on_chars(df, subdivision_size):
    rows = []

    for row in df.itertuples(index=False):
        title = row.title
        text = row.text

        for i in range(0, len(text), subdivision_size):
            chunk = text[i:i + subdivision_size]
            index_of_subtext = i // subdivision_size + 1

            rows.append({
                "title": title,
                "index_of_subtext": index_of_subtext,
                "text": chunk
            })

    return pd.DataFrame(rows)

In [4]:
not_cooking_df = pd.read_parquet("saved_data/dataset.parquet")
cooking_df = pd.read_parquet("saved_data/cooking_dataset.parquet")

In [5]:
cooking_df.head()

,title,text
0,morristowncookbo00vogt_djvu,"M& \n\n\nTheco°ur„ty SAVINGS BANK, \n\nCor. ..."
1,mushroombookpopu00marsrich_djvu,"/nr^T ; T T~~^ \n\nI . ,1 JL-a \n\n& \..."
2,buttfamilyfavori00unse_djvu,.Butt Family \nFavorites \n\n\nMf/liom Meet/...
3,usnavycookbook00instgoog_djvu,This is a digital copy of a book that ...
4,ourdailybread00scri_djvu,LIBRARY OF CONGRESS. \n\n\nShelf \n\n\nUNITE...


In [8]:
cooking_df = cooking_df[['title', 'text']]
not_cooking_df = not_cooking_df[['id', 'text']].iloc[:1000].rename(columns={'id': 'title'})

df = pd.concat([cooking_df, not_cooking_df], ignore_index=True)

In [9]:
df.describe()

,title,text
count,1506,1506
unique,1133,1133
top,41496-8,"The Project Gutenberg eBook, Addison, by Willi..."
freq,2,2


In [12]:
df["text"] = df["text"].apply(preprocess_text)

In [13]:
df.head()

,title,text
0,morristowncookbo00vogt_djvu,"m \n\nthecourty savings bank, \n\ncor. south a..."
1,mushroombookpopu00marsrich_djvu,"/nrt ; t t \n\ni . ,1 jl-a \n\n! i \n\nfrom th..."
2,buttfamilyfavori00unse_djvu,.butt family \nfavorites \n\nmf/liom meet/ act...
3,usnavycookbook00instgoog_djvu,this is a digital copy of a book that was pres...
4,ourdailybread00scri_djvu,library of congress. \n\nshelf \n\nunited stat...


In [14]:
df = divide_books_on_chars(df, subdivision_size=30000)

In [15]:
df.head()

,title,index_of_subtext,text
0,morristowncookbo00vogt_djvu,1,"m \n\nthecourty savings bank, \n\ncor. south a..."
1,morristowncookbo00vogt_djvu,2,lt and pepper and boil up once. \n\nmiss baldw...
2,morristowncookbo00vogt_djvu,3,"d colored in water \ncolors, or finished in mo..."
3,mushroombookpopu00marsrich_djvu,1,"/nrt ; t t \n\ni . ,1 jl-a \n\n! i \n\nfrom th..."
4,mushroombookpopu00marsrich_djvu,2,intersect to form a labyrinth of green networ...


In [16]:
df["index_of_subtext"].max()

np.int64(250)

In [11]:
df['is_cooking'] = [1]*len(cooking_df) + [0]*len(not_cooking_df)


KeyboardInterrupt: 

In [12]:
svm_model = SVC(probability=True)
logreg_model = LogisticRegression(solver='liblinear')
xgbmodel = XGBClassifier()


In [16]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = get_embeddings(df, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2046.05it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 48/48 [02:50<00:00,  3.55s/it]


In [18]:
df['mini_lm_embeddings'] = list(embeddings)

In [3]:
df.head()

NameError: name 'df' is not defined

In [24]:
xg = train_model(xgbmodel, df)

Model training completed. Test accuracy: 0.9934


In [25]:
df.head()

,title,text,mini_lm_embeddings,is_cooking
0,morristowncookbo00vogt_djvu,"M& \n\n\nTheco°ur„ty SAVINGS BANK, \n\nCor. ...","[0.038296927, 0.05159775, -0.117782675, 0.0166...",1
1,mushroombookpopu00marsrich_djvu,"/nr^T ; T T~~^ \n\nI . ,1 JL-a \n\n& \...","[0.024298232, 0.019099042, -0.020679126, -0.01...",1
2,buttfamilyfavori00unse_djvu,.Butt Family \nFavorites \n\n\nMf/liom Meet/...,"[-0.079372205, -0.03956904, -0.004220544, 0.04...",1
3,usnavycookbook00instgoog_djvu,This is a digital copy of a book that ...,"[-0.074365094, 0.038336515, -0.05250247, 0.005...",1
4,ourdailybread00scri_djvu,LIBRARY OF CONGRESS. \n\n\nShelf \n\n\nUNITE...,"[-0.056392536, 0.051913258, -0.058723737, 0.01...",1


In [26]:
svm = train_model(svm_model, df)

Model training completed. Test accuracy: 0.9934


In [27]:
logreg = train_model(logreg_model, df)

Model training completed. Test accuracy: 0.9934


In [ ]:

X = np.vstack(df['mini_lm_embeddings'].values) 
y = df['is_cooking'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=1000),
    'SVM': SVC(probability=True, kernel='linear'),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # probability of class 1

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    print(f"{name} -> Accuracy: {acc:.4f}, AUC: {auc:.4f}")

Logistic Regression -> Accuracy: 0.9967, AUC: 1.0000
SVM -> Accuracy: 1.0000, AUC: 1.0000
XGBoost -> Accuracy: 0.9868, AUC: 0.9998


/Users/yavuzlule/Desktop/bsc-thesis/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [07:16:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
file_path = "test_book_recipe_1.txt"
with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()

embedding = embedding_model.encode([text])  # returns a list of embeddings
for name, model in models.items():
    # 5️⃣ Predict probability
    proba = model.predict_proba(embedding)[0][1]  # probability of being cooking
    label = int(proba >= 0.5)

    print(f"Predicted label: {label} (1=cooking, 0=not cooking)")
    print(f"Probability of cooking: {proba:.4f}")

Predicted label: 1 (1=cooking, 0=not cooking)
Probability of cooking: 0.9131
Predicted label: 1 (1=cooking, 0=not cooking)
Probability of cooking: 1.0000
Predicted label: 1 (1=cooking, 0=not cooking)
Probability of cooking: 0.9972


In [34]:
for name, model in models.items():
    name = name.strip()
    save_model(model, "saved_models", name)

Model saved to saved_models/Logistic Regression.joblib
Model saved to saved_models/SVM.joblib
Model saved to saved_models/XGBoost.joblib


# Run models on chunks

In [ ]:
subdivsize = 7500

In [20]:
df = pd.read_parquet("saved_data/cooking_dataset.parquet")
df["text"] = df["text"].apply(preprocess_text)
df = divide_books_on_chars(df, subdivision_size=subdivsize)

In [7]:
trained_logreg = joblib.load("saved_models/logreg.joblib")
trained_svm = joblib.load("saved_models/SVM.joblib")
trained_xgb = joblib.load("saved_models/XGBoost.joblib")

In [21]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = get_embeddings(df, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 660.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 581/581 [05:04<00:00,  1.91it/s]


In [22]:
df['mini_lm_embeddings'] = list(embeddings)

In [23]:
df.head()

,title,index_of_subtext,text,mini_lm_embeddings
0,morristowncookbo00vogt_djvu,1,"m \n\nthecourty savings bank, \n\ncor. south a...","[0.047158398, 0.07172213, -0.113460466, -0.008..."
1,morristowncookbo00vogt_djvu,2,"umbs; \n\n24 \n\nfruit, vegetables, \n\nfish, ...","[-0.024562983, -0.061856054, 0.005260396, -0.0..."
2,morristowncookbo00vogt_djvu,3,"pan back \nfrom the fire, stir in gradually th...","[-0.021862814, 0.0035390004, 0.008252581, 0.02..."
3,morristowncookbo00vogt_djvu,4,"a small \n\nn \n\ngeo.w. melick, \n\ni \n\ni ...","[-0.016094353, -0.04163723, 0.011622076, -0.01..."
4,morristowncookbo00vogt_djvu,5,"and landscape \n\nviewing, groups, c. \n\nmin...","[-0.018920843, 0.07643922, 0.054061987, -0.001..."


In [24]:
X = np.vstack(df["mini_lm_embeddings"].values)

df["is_cooking_proba_logreg"] = trained_logreg.predict_proba(X)[:, 1]
df["is_cooking_proba_svm"] = trained_svm.predict_proba(X)[:, 1]
df["is_cooking_proba_xgb"] = trained_xgb.predict_proba(X)[:, 1]


In [25]:
df.head()

,title,index_of_subtext,text,mini_lm_embeddings,is_cooking_proba_logreg,is_cooking_proba_svm,is_cooking_proba_xgb
0,morristowncookbo00vogt_djvu,1,"m \n\nthecourty savings bank, \n\ncor. south a...","[0.047158398, 0.07172213, -0.113460466, -0.008...",0.804972,0.986642,0.994284
1,morristowncookbo00vogt_djvu,2,"umbs; \n\n24 \n\nfruit, vegetables, \n\nfish, ...","[-0.024562983, -0.061856054, 0.005260396, -0.0...",0.947463,0.997443,0.983422
2,morristowncookbo00vogt_djvu,3,"pan back \nfrom the fire, stir in gradually th...","[-0.021862814, 0.0035390004, 0.008252581, 0.02...",0.925858,0.993150,0.989925
3,morristowncookbo00vogt_djvu,4,"a small \n\nn \n\ngeo.w. melick, \n\ni \n\ni ...","[-0.016094353, -0.04163723, 0.011622076, -0.01...",0.952396,0.999999,0.999631
4,morristowncookbo00vogt_djvu,5,"and landscape \n\nviewing, groups, c. \n\nmin...","[-0.018920843, 0.07643922, 0.054061987, -0.001...",0.937133,0.999991,0.998058


In [26]:
df.to_parquet(f"saved_data/chunks_{subdivsize}_proba.parquet", index=False)

In [28]:
from pathlib import Path

directory = Path("archive-americana-cookbook-download-desc")

for file_path in directory.glob("*meta*"):
    if file_path.is_file():
        file_path.unlink()
        print(f"Deleted: {file_path}")